### Datasets to combine:
1. Departures Flight data
2. Weather data
3. FAA data

In [38]:
# Imports
from pathlib import Path

import gdown
import pandas as pd
import numpy as np

DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1SLMsPvHSMyiL9l5-lXEOh9OtPv11pVAU"

### Load CSV datasets from Google Drive Folder

In [39]:
SFO_flights_preprocessed = pd.read_csv("Final_Datasets/SFO_flights_preprocessed.csv")
FAA_merged_dataset = pd.read_csv("Final_Datasets/FAA_merged_dataset.csv")
sfo_noaa_cleaned = pd.read_csv("Final_Datasets/sfo_noaa_cleaned.csv")

### Merge Flights on FAA data - Anushka
1. Merge on Tail Number (Double Check Padding)

In [40]:
# Shape checks
print(SFO_flights_preprocessed.shape)
print(FAA_merged_dataset.shape)
print(sfo_noaa_cleaned.shape)

(444160, 33)
(313930, 17)
(38732, 23)


In [41]:
#Column names
#Need to stndardize these
print(SFO_flights_preprocessed.columns.tolist())
print(FAA_merged_dataset.columns.tolist())
print(sfo_noaa_cleaned.columns.tolist())

['carrier_code', 'date_mm_dd_yyyy', 'flight_number', 'tail_number', 'destination_airport', 'scheduled_departure_time', 'actual_departure_time', 'scheduled_elapsed_time_minutes', 'actual_elapsed_time_minutes', 'departure_delay_minutes', 'wheels_off_time', 'taxi_out_time_minutes', 'delay_carrier_minutes', 'delay_weather_minutes', 'delay_national_aviation_system_minutes', 'delay_security_minutes', 'delay_late_aircraft_arrival_minutes', 'is_cancelled', 'outcome', 'departure_delay_min', 'scheduled_dep_dt', 'actual_dep_dt', 'date', '6hr_cutoff_dt', 'year', 'month', 'month_name', 'day_of_week', 'is_weekend', 'sched_dep_hour', 'dep_time_of_day', 'is_holiday', 'mod_tail_number']
['N-NUMBER', 'YEAR MFR', 'TYPE AIRCRAFT', 'TYPE ENGINE', 'MFR MDL CODE', 'ENG MFR MDL', 'CODE', 'MFR', 'MODEL', 'NO-ENG', 'NO-SEATS', 'AC-WEIGHT', 'SPEED', 'CODE_ENG', 'MFR_ENG', 'MODEL_ENG', 'THRUST']
['DATE', 'Year', 'Month', 'Day', 'Hour', 'Minute', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'temperature', 'dew_point_temp

In [46]:
# Clean tail numbers for merging
SFO_flights_preprocessed["tail_number_clean"] = (
    SFO_flights_preprocessed["tail_number"]
    .astype("string")
    .str.strip()
    .str.upper()
    .str.replace(r"^N", "", regex=True)
)
FAA_merged_dataset["tail_number_clean"] = (
    FAA_merged_dataset["N-NUMBER"]
    .astype("string")
    .str.strip()
    .str.upper()
)


In [47]:
# Check duplicate aircraft records in FAA data
FAA_merged_dataset["tail_number_clean"].duplicated().sum()

0

In [48]:
# Merging flights data with FAA aircraft data
# Left join so we can keep all flight data records
flights_faa = SFO_flights_preprocessed.merge(
    FAA_merged_dataset,
    on="tail_number_clean",
    how="left"
)

In [33]:
# Check merge quality
print("Flights before merge:", len(SFO_flights_preprocessed))
print("Flights after merge:", len(flights_faa))

matched = flights_faa["N-NUMBER"].notna().sum()
print(f"Matched FAA records: {matched:,}")
print(f"Match rate: {100 * matched / len(flights_faa):.2f}%")

Flights before merge: 444160
Flights after merge: 444160
Matched FAA records: 423,890
Match rate: 95.44%


In [49]:
#unmatched examples for us to double check on the faa dataset to confirm they don't exist in there
flights_faa.loc[
    flights_faa["N-NUMBER"].isna(),
    ["tail_number", "tail_number_clean"]
].head(20)

,tail_number,tail_number_clean
4157,N823AW,823AW
17345,N823AW,823AW
28432,N823AW,823AW
43594,N823AW,823AW
43595,N823AW,823AW
45207,NaN,<NA>
45208,N636VA,636VA
45210,N927VA,927VA
45213,N845VA,845VA
45214,N839VA,839VA


In [50]:
# Checking if the tail number exists in the faa dataset
FAA_merged_dataset[
    FAA_merged_dataset["tail_number_clean"] == "823AW"
]

,N-NUMBER,YEAR MFR,TYPE AIRCRAFT,TYPE ENGINE,MFR MDL CODE,ENG MFR MDL,CODE,MFR,MODEL,NO-ENG,NO-SEATS,AC-WEIGHT,SPEED,CODE_ENG,MFR_ENG,MODEL_ENG,THRUST,tail_number_clean


In [51]:
# Drop flights without FAA aircraft information
flights_faa = flights_faa.dropna(subset=["N-NUMBER"]).copy()

print(f"Remaining flights: {len(flights_faa):,}")
print(f"Dropped flights: {len(SFO_flights_preprocessed) - len(flights_faa):,}")

Remaining flights: 423,890
Dropped flights: 20,270


### Clean Flight/FAA merged data
1. Flights without FAA data --> Drop

In [36]:
#Standardizing column names 
flights_faa.columns = (
    flights_faa.columns
    .str.strip()
    .str.lower()
    .str.replace(r'[\s\(\)/\-]+', '_', regex=True)
    .str.replace(r'[^\w]', '', regex=True)
    .str.strip('_')
)

print(flights_faa.columns.tolist())
flights_faa.head()

['carrier_code', 'date_mm_dd_yyyy', 'flight_number', 'tail_number', 'destination_airport', 'scheduled_departure_time', 'actual_departure_time', 'scheduled_elapsed_time_minutes', 'actual_elapsed_time_minutes', 'departure_delay_minutes', 'wheels_off_time', 'taxi_out_time_minutes', 'delay_carrier_minutes', 'delay_weather_minutes', 'delay_national_aviation_system_minutes', 'delay_security_minutes', 'delay_late_aircraft_arrival_minutes', 'is_cancelled', 'outcome', 'departure_delay_min', 'scheduled_dep_dt', 'actual_dep_dt', 'date', '6hr_cutoff_dt', 'year', 'month', 'month_name', 'day_of_week', 'is_weekend', 'sched_dep_hour', 'dep_time_of_day', 'is_holiday', 'mod_tail_number', 'tail_number_clean', 'n_number', 'year_mfr', 'type_aircraft', 'type_engine', 'mfr_mdl_code', 'eng_mfr_mdl', 'code', 'mfr', 'model', 'no_eng', 'no_seats', 'ac_weight', 'speed', 'code_eng', 'mfr_eng', 'model_eng', 'thrust']


,carrier_code,date_mm_dd_yyyy,flight_number,tail_number,destination_airport,scheduled_departure_time,actual_departure_time,scheduled_elapsed_time_minutes,actual_elapsed_time_minutes,departure_delay_minutes,...,mfr,model,no_eng,no_seats,ac_weight,speed,code_eng,mfr_eng,model_eng,thrust
0,AA,01/01/2022,16.0,N110AN,JFK,11:46,11:35,343.0,328.0,-11.0,...,AIRBUS INDUSTRIE,A321-231,2.0,379.0,CLASS 3,0.0,34611.0,IAE,V2533-A5,31600.0
1,AA,01/01/2022,234.0,N112AN,JFK,07:00,06:54,329.0,328.0,-6.0,...,AIRBUS INDUSTRIE,A321-231,2.0,379.0,CLASS 3,0.0,34611.0,IAE,V2533-A5,31600.0
2,AA,01/01/2022,276.0,N109NN,JFK,22:59,22:51,333.0,329.0,-8.0,...,AIRBUS INDUSTRIE,A321-231,2.0,379.0,CLASS 3,0.0,34611.0,IAE,V2533-A5,31600.0
3,AA,01/01/2022,369.0,N341SV,MIA,07:47,07:43,330.0,303.0,-4.0,...,BOEING,737-8,2.0,175.0,CLASS 3,0.0,13134.0,CFM INTL,LEAP-1B25,26786.0
4,AA,01/01/2022,1175.0,N992AN,DFW,17:21,17:31,208.0,195.0,10.0,...,BOEING,737-823,2.0,162.0,CLASS 3,0.0,13074.0,CFM INTL,CFM56-7B24/3,22800.0


In [37]:
print("Shape of flights df:", flights_faa.shape)
print("Data types:")
print(flights_faa.dtypes)
flights_faa.head()

Shape of flights df: (444160, 51)
Data types:
carrier_code                                      object
date_mm_dd_yyyy                                   object
flight_number                                    float64
tail_number                                       object
destination_airport                               object
scheduled_departure_time                          object
actual_departure_time                             object
scheduled_elapsed_time_minutes                   float64
actual_elapsed_time_minutes                      float64
departure_delay_minutes                          float64
wheels_off_time                                   object
taxi_out_time_minutes                            float64
delay_carrier_minutes                            float64
delay_weather_minutes                            float64
delay_national_aviation_system_minutes           float64
delay_security_minutes                           float64
delay_late_aircraft_arrival_minutes       

,carrier_code,date_mm_dd_yyyy,flight_number,tail_number,destination_airport,scheduled_departure_time,actual_departure_time,scheduled_elapsed_time_minutes,actual_elapsed_time_minutes,departure_delay_minutes,...,mfr,model,no_eng,no_seats,ac_weight,speed,code_eng,mfr_eng,model_eng,thrust
0,AA,01/01/2022,16.0,N110AN,JFK,11:46,11:35,343.0,328.0,-11.0,...,AIRBUS INDUSTRIE,A321-231,2.0,379.0,CLASS 3,0.0,34611.0,IAE,V2533-A5,31600.0
1,AA,01/01/2022,234.0,N112AN,JFK,07:00,06:54,329.0,328.0,-6.0,...,AIRBUS INDUSTRIE,A321-231,2.0,379.0,CLASS 3,0.0,34611.0,IAE,V2533-A5,31600.0
2,AA,01/01/2022,276.0,N109NN,JFK,22:59,22:51,333.0,329.0,-8.0,...,AIRBUS INDUSTRIE,A321-231,2.0,379.0,CLASS 3,0.0,34611.0,IAE,V2533-A5,31600.0
3,AA,01/01/2022,369.0,N341SV,MIA,07:47,07:43,330.0,303.0,-4.0,...,BOEING,737-8,2.0,175.0,CLASS 3,0.0,13134.0,CFM INTL,LEAP-1B25,26786.0
4,AA,01/01/2022,1175.0,N992AN,DFW,17:21,17:31,208.0,195.0,10.0,...,BOEING,737-823,2.0,162.0,CLASS 3,0.0,13074.0,CFM INTL,CFM56-7B24/3,22800.0


### Flight/FAA - Feature Selection
1. Decide on columns to keep (not inside 6 hour window before departure).

### Merge Flights/FAA data on Weather data
1. Using merge_asof
- Using nearest hourly timestamp after 6 hour time window
- Using 6 hour cutoff timestamp

### Clean Flights/Weather merged data
1. Removing rows with timestamp gap of more than an hour (e.g. 6 hour cutoff: 11:30am)
- good nearest weather timestamp: 10:56am
- bad nearest weather timestamp: <= 9:56am


### Weather - Feature Selection

### Final Clean
1. Split train, validate, and test data - perform EDA on train only
2. Summarize nan values
3. Remove redundant features with 1 to 1 correlation
    1. Refer to Top 6 of Missingness Heatmap